In [34]:
## imports

import os
import yaml
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import  MarkdownHeaderTextSplitter

In [2]:
currentDirectory = os.getcwd()
currentDirectory

'c:\\Users\\gabri\\Documents\\capstone-projects\\airline-support-bot\\backend\\rag-service\\notebooks'

In [25]:
## load data from the knowledge base 
loader = DirectoryLoader("../data/raw/",glob="**/*.md",loader_cls=TextLoader)
documents = loader.load()

print(f"number of documents :{len(documents)}")


number of documents :30


In [29]:
print(f"document one:{documents[0]}")
print("Content:\n", documents[0].page_content)
print("Metadata:\n", documents[0].metadata)


document one:page_content='

# Airport Lounge Services

Kenya Airways operates premium lounges at Jomo Kenyatta International Airport (JKIA) that provide passengers with a comfortable environment before departure. The lounges offer services designed to improve the airport experience, including dining facilities, relaxation areas, business facilities, and passenger amenities.

## Kenya Airways Lounges

Kenya Airways operates the following lounges:

- Pride Lounge
- Simba Lounge
- Asante Lounge
- Msafiri Lounge

## Lounge Access Eligibility

Passengers eligible for Kenya Airways lounge access include:

- Kenya Airways Business Class passengers.
- SkyTeam Platinum and Gold cardholders.
- Eligible passengers travelling on Kenya Airways partner airlines.
- Economy Class passengers may purchase lounge access subject to availability.

## Lounge Facilities

Kenya Airways lounges provide various facilities including:

- Comfortable seating areas.
- High-speed Wi-Fi access.
- Food and beverage s

In [ ]:
## adding extra metadata to the documents
for document in documents:
    content = document.page_content

    # Split the front matter from the Markdown content
    if content.startswith("---"):
        _, front_matter, markdown = content.split("---", 2)

        # Convert YAML front matter into a Python dictionary
        metadata = yaml.safe_load(front_matter)

        # Add your metadata to LangChain's existing metadata
        document.metadata.update(metadata)

        # Remove the metadata from the actual page content
        document.page_content = markdown

In [37]:
## creating chunks from the documents
## 1. Creating the text splitter
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "section"),
        ("##", "subsection"),
        ("###", "subsubsection")
    ]
)

recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [38]:

chunks = []

for document in documents:

    # First split according to Markdown headings
    sections = markdown_splitter.split_text(
        document.page_content
    )

    for section in sections:

        # Carry the original document metadata into the section
        section.metadata.update(document.metadata)

        # Split the section further if it is too large
        smaller_chunks = recursive_splitter.split_documents(
            [section]
        )

        for chunk in smaller_chunks:

            # Give every final chunk a unique ID
            chunk.metadata["chunk_id"] = (
                f"{document.metadata.get('document_id', 'UNKNOWN')}"
                f"-chunk-{len(chunks) + 1:03d}"
            )

            chunks.append(chunk)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 279


In [40]:
print(chunks[3].page_content)
print(chunks[3].metadata)

Kenya Airways lounges provide various facilities including:  
- Comfortable seating areas.
- High-speed Wi-Fi access.
- Food and beverage services.
- Buffet meals and snacks.
- Live cooking services at selected lounges.
- Shower facilities.
- Television entertainment.
- Business facilities with workspace areas.
- Charging points for electronic devices.
{'section': 'Airport Lounge Services', 'subsection': 'Lounge Facilities', 'source': '..\\data\\raw\\airport_services\\lounge_services.md', 'document_id': 'KQ-AIRPORT-001', 'title': 'Airport Lounge Services', 'origin': 'KENYA AIRWAYS', 'domain': 'AIRPORT_SERVICES', 'category': 'LOUNGE_SERVICES', 'document_type': 'POLICY', 'applicable_to': ['CUSTOMER', 'CUSTOMER_SERVICE_AGENT'], 'access': 'PUBLIC', 'status': 'ACTIVE', 'language': 'EN', 'chunk_id': 'KQ-AIRPORT-001-chunk-004'}
